# Experiment 2b — Conflict behavioral test on held-out evolved pairs

In-distribution generalization test for the exp1b authority probe. Uses pairs from
the UltraFeedback-evolved conflict dataset that were NOT in the stratified 1k used
for training. Generation is done with the HF model (so we can hook per-layer probe
scores during the prefill); compliance is judged by a vLLM-served instruct model
(loaded after the HF model is freed).

Requires an A100 (80GB recommended — fits a 7-9B HF model + a 7B vLLM judge sequentially).

In [ ]:
# Install
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}

!pip install -q -e {REPO_DIR}
!pip install -q vllm

os.environ["MECH_SPOOF_ROOT"] = REPO_DIR
src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()

import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
import os
from google.colab import drive, userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
drive.mount("/content/drive")

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = "qwen"            # qwen | llama3 | mistral | gemma | phi3 | gemma_small
DRIVE_ROOT = Path("/content/drive/MyDrive/mech_spoof_results")

# Probe (from exp1b)
PROBE_DIR_NAME = "exp1b_authority_conflict"
PROBE_POSITION = None         # None = use bundle's best_position; or 'response_first' / 'response_mid' / 'response_last'

# Held-out sample size and judge model
N_PAIRS = 200
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # any chat-tuned vLLM-compatible model
JUDGE_BATCH_SIZE = 64

slug = MODEL_CONFIGS[MODEL_KEY].slug
OUT_DIR = DRIVE_ROOT / slug / "exp2b_conflict_evolved"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PROBE_DIR = DRIVE_ROOT / slug / PROBE_DIR_NAME

print(f"Model:       {MODEL_KEY} ({MODEL_CONFIGS[MODEL_KEY].hf_id})")
print(f"OUT_DIR    = {OUT_DIR}")
print(f"PROBE_DIR  = {PROBE_DIR}  (exists={PROBE_DIR.exists()})")
print(f"JUDGE      = {JUDGE_MODEL_ID}")
print(f"N_PAIRS    = {N_PAIRS}")

In [ ]:
# Run experiment 2b
from mech_spoof.experiments.exp2b_conflict_evolved import run_experiment_2b

result = run_experiment_2b(
    MODEL_KEY,
    OUT_DIR,
    exp1_dir=PROBE_DIR,
    probe_position=PROBE_POSITION,
    judge_model_id=JUDGE_MODEL_ID,
    n_pairs=N_PAIRS,
    batch_size=8,                 # HF generation batch
    judge_batch_size=JUDGE_BATCH_SIZE,
    max_new_tokens=512,
)
print(f"\nDone. n_judged={result.n_judged}/{result.n_pairs * 3}")

## Results

In [ ]:
# Summary by condition
print(f"Probe position used: {result.probe_position}")
print(f"Judge model:         {result.judge_model_id}\n")
print(f"{'cond':>6s}  {'n':>5s}  {'judged':>6s}  {'comply':>6s}  {'mean_ps':>8s}  {'std_ps':>8s}")
print("-" * 50)
for cond, s in result.summary.items():
    print(
        f"{cond:>6s}  {s['n']:5d}  {s['n_judged']:6d}  "
        f"{s['compliance_rate'] if s['compliance_rate'] is not None else float('nan'):6.3f}  "
        f"{s['mean_probe_score'] if s['mean_probe_score'] is not None else float('nan'):8.3f}  "
        f"{s['std_probe_score'] if s['std_probe_score'] is not None else float('nan'):8.3f}"
    )

In [ ]:
# Best-layer correlations
import json
for k in ("overall", "REAL", "NONE", "FAKE"):
    if k in result.correlation:
        c = result.correlation[k]
        print(f"{k:>8s}: r={c['r']:+.3f}  p={c['p']:.3g}  n={c['n']}")
    else:
        print(f"{k:>8s}: (insufficient variance / N)")

In [ ]:
# Per-layer Pearson r curve (overall + per condition)
import matplotlib.pyplot as plt
by_layer = result.correlation.get("by_layer", {})
if not by_layer:
    print("No per-layer correlations recorded.")
else:
    layers = sorted(int(l) for l in by_layer.keys())
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, color in (("overall", None), ("REAL", None), ("NONE", None), ("FAKE", None)):
        ys = [by_layer.get(l, {}).get(label, {}).get("r", float("nan")) for l in layers]
        if any(y == y for y in ys):  # any non-NaN
            ax.plot(layers, ys, marker=".", label=label)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Pearson r (probe score vs system_followed)")
    ax.set_title(f"{MODEL_KEY}: exp2b per-layer correlation (held-out evolved)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "per_layer_correlation.png", dpi=120)
    plt.show()

In [ ]:
# Spot-check a few judged rows
from mech_spoof.io import load_json
rows = load_json(OUT_DIR / "result.json")["rows"]
import random
random.seed(0)
for r in random.sample(rows, min(5, len(rows))):
    print(f"--- pair {r['pair_idx']} | {r['condition']} | axis={r['conflict_axis']} ---")
    print(f"S: {r['s_instruction'][:100]}")
    print(f"U: {r['u_instruction'][:100]}")
    print(f"verdict={r['judge_verdict']}  followed_system={r['system_followed']}  probe={r['probe_score']}")
    print(f"reason: {r['judge_reason'][:160]}")
    print(f"resp:   {r['response'][:240]}")
    print()